[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/12_gradient_boosting/03_xgboost_lightgbm_catboost.ipynb)

# Bài 03 — XGBoost, LightGBM, CatBoost và HistGradientBoosting

**🎯 Mục tiêu**
- Hiểu "bí quyết" của XGBoost: **khai triển Taylor bậc hai** + **regularization** → công thức đóng cho giá trị lá $w^* = -\frac{G}{H + \lambda}$ và **gain** của split.
- Các cải tiến tốc độ: **histogram** (binning), **leaf-wise growth** (LightGBM), GOSS/EFB; **ordered boosting** + xử lý categorical (CatBoost).
- Dùng thực tế: `HistGradientBoostingClassifier` (sklearn), XGBoost, LightGBM, CatBoost — early stopping, missing values, feature phân loại.
- So sánh tốc độ và chất lượng trên cùng một bài toán.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/12_gradient_boosting

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()

# Các thư viện boosting là tuỳ chọn — notebook vẫn chạy nếu thiếu
try:
    import xgboost as xgb
except ImportError:
    xgb = None
try:
    import lightgbm as lgb
except ImportError:
    lgb = None
try:
    import catboost as cb
except ImportError:
    cb = None
print({"xgboost": xgb is not None, "lightgbm": lgb is not None, "catboost": cb is not None})
warnings.filterwarnings("ignore", message=".*eval_set.*")      # API eval_set của LightGBM đang đổi tên giữa các phiên bản

## 1. XGBoost: Taylor bậc hai + regularization

Ở vòng $m$, cần thêm cây $f$ để tối thiểu $\sum_i L(y_i, F_{m-1}(x_i) + f(x_i)) + \Omega(f)$ với $\Omega(f) = \gamma T + \frac{\lambda}{2}\sum_j w_j^2$ ($T$ lá, giá trị lá $w_j$). Khai triển Taylor bậc hai với $g_i = \partial_F L$, $h_i = \partial^2_F L$:

$$\tilde L = \sum_i \left[g_i f(x_i) + \tfrac12 h_i f(x_i)^2\right] + \gamma T + \tfrac{\lambda}{2}\sum_j w_j^2 = \sum_{j=1}^T\left[G_j w_j + \tfrac12(H_j + \lambda)w_j^2\right] + \gamma T$$

với $G_j = \sum_{i \in \text{lá } j} g_i$, $H_j = \sum_{i \in \text{lá } j} h_i$. Mỗi lá là một parabol độc lập → **nghiệm đóng**:

$$w_j^* = -\frac{G_j}{H_j + \lambda}, \qquad \tilde L^* = -\frac12\sum_j \frac{G_j^2}{H_j + \lambda} + \gamma T$$

**Gain** của một split (trái $L$, phải $R$):

$$\text{Gain} = \frac12\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right] - \gamma$$

→ $\lambda$ co giá trị lá về 0, $\gamma$ là ngưỡng gain tối thiểu (pruning). Với log-loss: $g_i = p_i - y_i$, $h_i = p_i(1 - p_i)$.

In [ ]:
def xgb_leaf_and_gain(g, h, mask, lam=1.0, gamma=0.0):
    G_L, H_L = g[mask].sum(), h[mask].sum()
    G_R, H_R = g[~mask].sum(), h[~mask].sum()
    gain = 0.5 * (G_L**2 / (H_L + lam) + G_R**2 / (H_R + lam) - (G_L + G_R) ** 2 / (H_L + H_R + lam)) - gamma
    return -G_L / (H_L + lam), -G_R / (H_R + lam), gain


# Ví dụ tính tay: 6 mẫu, mọi dự đoán ban đầu p = 0.5 (F = 0)
y_toy = np.array([1, 1, 1, 0, 0, 1])
x_toy = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
p = np.full(6, 0.5)
g, h = p - y_toy, p * (1 - p)
for t in [2.5, 3.5, 4.5, 5.5]:
    wl, wr, gain = xgb_leaf_and_gain(g, h, x_toy <= t, lam=1.0)
    print(f"split x <= {t}: w_trái = {wl:+.3f}, w_phải = {wr:+.3f}, gain = {gain:.4f}")

Split $x \le 3.5$ có gain lớn nhất: bên trái toàn nhãn 1 → $w > 0$ (tăng log-odds), bên phải chủ yếu nhãn 0 → $w < 0$. Lưu ý $\lambda = 1$ làm $|w|$ nhỏ hơn so với không regularize.

## 2. Các cải tiến tốc độ

| Kỹ thuật | Thư viện | Ý tưởng |
|---|---|---|
| **Histogram binning** | LightGBM, XGBoost `hist`, sklearn `HistGradientBoosting` | gom giá trị feature vào ≤ 255 bin → tìm split $O(\text{bins})$ thay vì $O(n)$; dùng hiệu histogram cha − con |
| **Leaf-wise growth** | LightGBM | luôn tách **lá có gain lớn nhất** (thay vì tách hết theo tầng) → ít lá hơn cho cùng loss; điều khiển bằng `num_leaves` |
| **GOSS** | LightGBM | giữ mẫu có gradient lớn, lấy mẫu ngẫu nhiên mẫu gradient nhỏ |
| **EFB** | LightGBM | gộp các feature thưa không bao giờ cùng khác 0 |
| **Ordered boosting** | CatBoost | tính residual cho mỗi mẫu bằng model **không** thấy mẫu đó → giảm "prediction shift" (rò rỉ target) |
| **Ordered target statistics** | CatBoost | mã hoá categorical bằng thống kê target tính theo thứ tự ngẫu nhiên — chống leakage |
| **Missing values** | cả bốn | học hướng mặc định (trái/phải) cho giá trị thiếu ở mỗi split |

## 3. So sánh trên dữ liệu churn (có feature phân loại và missing)

In [ ]:
churn = make_customer_churn(n=20000, seed=21)
rng = np.random.default_rng(0)
churn.loc[rng.random(len(churn)) < 0.1, "monthly_charges"] = np.nan          # 10% missing
cat_cols = ["contract", "internet_service", "payment_method"]
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products", "senior"]
X = churn[num_cols + cat_cols].copy()
X["senior"] = X["senior"].astype(int)
for c in cat_cols:
    X[c] = X[c].astype("category")
y = churn["churn"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=0, stratify=y_tr)
X_onehot_tr = pd.get_dummies(X_tr, dtype=float).fillna(-1)                     # sklearn GB cũ không nhận NaN/category
X_onehot_te = pd.get_dummies(X_te, dtype=float).fillna(-1)
print(X_tr.shape, "| tỉ lệ churn:", y.mean().round(3))

In [ ]:
results = []


def record(name, fit_fn, predict_fn):
    start = time.perf_counter()
    fit_fn()
    t = time.perf_counter() - start
    p = predict_fn()
    results.append({"model": name, "thời gian train (s)": round(t, 2),
                    "ROC-AUC": round(roc_auc_score(y_te, p), 4), "log-loss": round(log_loss(y_te, p), 4)})


gbc = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=0)
record("sklearn GradientBoosting (one-hot)", lambda: gbc.fit(X_onehot_tr, y_tr), lambda: gbc.predict_proba(X_onehot_te)[:, 1])

hgb = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, categorical_features="from_dtype",
                                     early_stopping=True, validation_fraction=0.2, n_iter_no_change=30, random_state=0)
record("sklearn HistGradientBoosting", lambda: hgb.fit(X_tr, y_tr), lambda: hgb.predict_proba(X_te)[:, 1])

if xgb is not None:
    xgbm = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, tree_method="hist",
                             enable_categorical=True, early_stopping_rounds=50, eval_metric="logloss", n_jobs=2)
    record("XGBoost", lambda: xgbm.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False),
           lambda: xgbm.predict_proba(X_te)[:, 1])
if lgb is not None:
    lgbm = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=15, verbose=-1, n_jobs=2)
    record("LightGBM", lambda: lgbm.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)]),
           lambda: lgbm.predict_proba(X_te)[:, 1])
if cb is not None:
    cbm = cb.CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=4, verbose=0, thread_count=2, allow_writing_files=False,
                                cat_features=cat_cols, early_stopping_rounds=50, random_seed=0)
    X_fit_cb, X_val_cb, X_te_cb = (d.astype({c: str for c in cat_cols}) for d in (X_fit, X_val, X_te))
    record("CatBoost", lambda: cbm.fit(X_fit_cb, y_fit, eval_set=(X_val_cb, y_val)), lambda: cbm.predict_proba(X_te_cb)[:, 1])

pd.DataFrame(results)

Trên bài toán này các thư viện boosting hiện đại cho chất lượng **tương đương nhau** — chênh lệch nhỏ hơn nhiều so với chênh lệch do feature engineering hay tuning. Khác biệt chính nằm ở **tốc độ** (histogram nhanh hơn hẳn GB cổ điển khi dữ liệu lớn), **tiện dụng** (categorical, missing, early stopping tích hợp) và **hệ sinh thái** (GPU, phân tán, SHAP).

## 4. Tốc độ theo kích thước dữ liệu: exact vs histogram

In [ ]:
sizes = [2_000, 10_000, 40_000]
times = {"GradientBoosting (exact)": [], "HistGradientBoosting": []}
big = make_customer_churn(n=max(sizes), seed=3)
Xb = pd.get_dummies(big.drop(columns=["customer_id", "churn"]), dtype=float).to_numpy()
yb = big["churn"].to_numpy()
for n in sizes:
    for name, model in [("GradientBoosting (exact)", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)),
                        ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=100, max_depth=3, early_stopping=False, random_state=0))]:
        start = time.perf_counter()
        model.fit(Xb[:n], yb[:n])
        times[name].append(time.perf_counter() - start)
for name, t in times.items():
    plt.plot(sizes, t, "o-", label=name)
plt.xlabel("số mẫu train")
plt.ylabel("thời gian train (s), 100 cây")
plt.legend()
plt.title("Histogram binning: tăng chậm hơn nhiều theo kích thước dữ liệu")
plt.show()
for i, n in enumerate(sizes):
    print(f"n = {n:6d}: exact {times['GradientBoosting (exact)'][i]:.2f}s | histogram {times['HistGradientBoosting'][i]:.2f}s")

## 5. Một vài tham số tương ứng giữa các thư viện

| Ý nghĩa | XGBoost | LightGBM | CatBoost | sklearn Hist GB |
|---|---|---|---|---|
| số cây | `n_estimators` | `n_estimators` | `iterations` | `max_iter` |
| learning rate | `learning_rate` (`eta`) | `learning_rate` | `learning_rate` | `learning_rate` |
| độ phức tạp cây | `max_depth` | `num_leaves` (+ `max_depth`) | `depth` (cây đối xứng) | `max_leaf_nodes`, `max_depth` |
| tối thiểu ở lá | `min_child_weight` (tổng hessian) | `min_child_samples` | `min_data_in_leaf` | `min_samples_leaf` |
| L2 trên giá trị lá | `reg_lambda` | `reg_lambda` | `l2_leaf_reg` | `l2_regularization` |
| lấy mẫu hàng / cột | `subsample`, `colsample_bytree` | `bagging_fraction`, `feature_fraction` | `subsample`, `rsm` | — / `max_features` |

## 🧠 Kiểm tra nhanh

1. Một lá có $G = -12$, $H = 5$, $\lambda = 1$. Giá trị lá $w^*$? Nếu $\lambda = 10$?
2. Vì sao XGBoost cần đạo hàm bậc hai của loss? Lợi ích so với GB cổ điển?
3. Leaf-wise growth của LightGBM khác level-wise thế nào? Rủi ro?
4. CatBoost làm gì để mã hoá categorical mà không leakage?

## 📌 Tóm tắt

- XGBoost: Taylor bậc hai + regularization → $w^* = -G/(H + \lambda)$, gain có công thức đóng, $\gamma$ để pruning.
- Histogram + leaf-wise (LightGBM), ordered boosting + categorical (CatBoost), HistGradientBoosting (sklearn) — nhanh, xử lý missing/categorical.
- Chất lượng tương đương nhau khi tune hợp lý — chọn theo tốc độ, tiện dụng, hệ sinh thái.

## 📚 Tài liệu tham khảo

- Chen & Guestrin (2016) — *XGBoost: A Scalable Tree Boosting System*
- Ke et al. (2017) — *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*
- Prokhorenkova et al. (2018) — *CatBoost: unbiased boosting with categorical features*